In [1]:
import pandas as pd
import numpy as np

# Load clean dataset 
df = pd.read_csv("pharmaguard_clean.csv", low_memory=False)
print(f"Clean dataset loaded: {len(df):,} rows")
print(f"\nBasic statistics:")
print(df[['total_claims', 'total_cost', 'total_patients', 'total_days_supply']].describe())

Clean dataset loaded: 89,710 rows

Basic statistics:
       total_claims    total_cost  total_patients  total_days_supply
count  89710.000000  8.971000e+04    89710.000000       89710.000000
mean      89.880203  8.912743e+03       31.349827        4818.168710
std      143.041140  6.052608e+04       77.611906        7324.362354
min       11.000000  6.680000e+00       11.000000          11.000000
25%       31.000000  3.679675e+02       14.000000         780.000000
50%       54.000000  9.358250e+02       20.000000        2805.000000
75%      101.000000  2.371985e+03       34.000000        5577.000000
max    20788.000000  6.698267e+06    20783.000000      198021.000000


Fraud Injection

In [2]:
# Fraud Injection
np.random.seed(42)

df['is_fraud'] = 0
df['fraud_type'] = 'legitimate'

N = len(df)

Fraud Type 1: Volume Fraud

In [3]:
# Prescribers with impossibly high claim volumes
n_volume = int(N * 0.02)  # 2% of dataset
volume_idx = np.random.choice(df.index, n_volume, replace=False)
df.loc[volume_idx, 'total_claims'] = np.random.randint(5000, 20000, n_volume)
df.loc[volume_idx, 'total_patients'] = np.random.randint(1000, 5000, n_volume)
df.loc[volume_idx, 'is_fraud'] = 1
df.loc[volume_idx, 'fraud_type'] = 'volume_fraud'

Fraud Type 2: Cost Fraud

In [4]:
# Abnormally high cost per patient compared to drug average
n_cost = int(N * 0.02)  # 2% of dataset
remaining = df[df['is_fraud'] == 0].index
cost_idx = np.random.choice(remaining, n_cost, replace=False)
df.loc[cost_idx, 'total_cost'] = np.random.uniform(500000, 6000000, n_cost)
df.loc[cost_idx, 'total_claims'] = np.random.randint(10, 50, n_cost)
df.loc[cost_idx, 'is_fraud'] = 1
df.loc[cost_idx, 'fraud_type'] = 'cost_fraud'

Fraud Type 3: Off-label Proxy 

In [5]:
# Specialty incompatible with drug at high volume
off_label_pairs = {
    'Dermatology': ['Metformin', 'Lisinopril', 'Atorvastatin', 'Warfarin'],
    'Ophthalmology': ['Metformin', 'Amoxicillin', 'Gabapentin', 'Oxycodone'],
    'Dentist': ['Warfarin', 'Insulin', 'Metformin', 'Lisinopril'],
}

remaining = df[df['is_fraud'] == 0].index
n_offlabel = int(N * 0.02)  # 2% of dataset
off_idx = np.random.choice(remaining, n_offlabel, replace=False)

for idx in off_idx:
    specialty = df.loc[idx, 'specialty']
    if specialty in off_label_pairs:
        df.loc[idx, 'generic_name'] = np.random.choice(off_label_pairs[specialty])
    df.loc[idx, 'total_claims'] = np.random.randint(300, 1000, 1)[0]
    df.loc[idx, 'is_fraud'] = 1
    df.loc[idx, 'fraud_type'] = 'off_label'

Summary

In [6]:
# Summary 
print(f"Total rows: {len(df):,}")
print(f"\nFraud distribution:")
print(df['fraud_type'].value_counts())
print(f"\nFraud percentage: {df['is_fraud'].mean()*100:.1f}%")

# Save
df.to_csv("pharmaguard_fraud.csv", index=False)
print(f"\nFile saved: pharmaguard_fraud.csv")


Total rows: 89,710

Fraud distribution:
fraud_type
legitimate      84328
off_label        1794
cost_fraud       1794
volume_fraud     1794
Name: count, dtype: int64

Fraud percentage: 6.0%

File saved: pharmaguard_fraud.csv
